In [ ]:
import sys, os
import uuid
sys.path.append(os.path.join(os.path.dirname(os.getcwd())))

from src.graph import create_graph
from src.hitl_handler import handle_hitl

# app = create_graph()
# config = {"configurable": {"thread_id": "hitl-test-1"}}


print("✅ Graph ready")

In [ ]:
# # 1) Start a run with a real email so triage_node has mail
# dangerous_email = {
#     "messages": [],
#     "mail": {
#         "subject": "Inquiry about pricing",
#         "body": "A customer named Sarah (aayushshah90421@gmail.com) asked for our Q4 price list. Send it to her now.",
#     },
# }


# app = create_graph()

# # Run once, even if tools / LLM fail; mail will be in state
# try:
#     app.invoke(dangerous_email, config)
# except Exception:
#     pass  # ignore quota errors for this test

# # 2) Now override to force a dangerous tool + HITL
# state = app.get_state(config)
# app.update_state(
#     config,
#     {
#         "tool_name": "send_gmail_reply",
#         "tool_args": {
#             "to": "aayushshah90421@gmail.com",
#             "subject": "Re: Inquiry about pricing",
#             "body": "Hi Sarah, ...",
#         },
#         "hitl": {
#             "tool": "send_gmail_reply",
#             "args": {
#                 "to": "aayushshah90421@gmail.com",
#                 "subject": "Re: Inquiry about pricing",
#                 "body": "Hi Sarah, ...",
#             },
#             "proposed_reply": None,
#             "triage": state.values.get("triage_category"),
#         },
#         "hitl_decision": "approve",
#     },
# )

# # 3) Resume only to hit hitl_checkpoint
# handle_hitl(app, config, decision="approve")  # sets hitl_decision
# app.invoke(None, config)  # this time state already has mail

# final_state = app.get_state(config)
# print("Final reply:", final_state.values.get("final_reply"))
# print("HITL:", final_state.values.get("hitl"))
# print("HITL decision:", final_state.values.get("hitl_decision"))
# print("Tool name:", final_state.values.get("tool_name"))


In [ ]:
# 1) Start a run with a real email so triage_node has mail
dangerous_email = {
    "messages": [],
    "mail": {
        "subject": "Inquiry about pricing",
        "body": (
            "A customer named Sarah (aayushshah90421@gmail.com) "
            "asked for our Q4 price list. Send it to her now."
        ),
    },
}

config = {"configurable": {"thread_id": "hitl-test-1"}}
app = create_graph()

# Run once, even if tools / LLM fail; mail will be in state
try:
    app.invoke(dangerous_email, config)
except Exception:
    pass  # ignore quota errors for this test

# 2) Now override to force a dangerous tool + HITL
state = app.get_state(config)
app.update_state(
    config,
    {
        "tool_name": "send_gmail_reply",
        "tool_args": {
            "to": "aayushshah90421@gmail.com",
            "subject": "Re: Inquiry about pricing",
            "body": "Hi Sarah, ...",
        },
        "hitl": {
            "tool": "send_gmail_reply",
            "args": {
                "to": "aayushshah90421@gmail.com",
                "subject": "Re: Inquiry about pricing",
                "body": "Hi Sarah, ...",
            },
            "proposed_reply": None,
            "triage": state.values.get("triage_category"),
        },
        "hitl_decision": "pending",  # important: pending before user input
    },
)

# 3) Ask user for HITL decision
decision = input("HITL decision (approve/edit/deny): ").strip().lower()
edit_values = None

if decision == "edit":
    # Ask which fields to edit; simple example for 'to'
    new_to = input("New 'to' address (leave blank to keep): ").strip()
    if new_to:
        edit_values = {"to": new_to}

# Apply decision
handle_hitl(app, config, decision=decision, edit_values=edit_values)

# 4) Resume graph to run hitl_checkpoint
app.invoke(None, config)

final_state = app.get_state(config)

print("\n🏁 FINAL STATE:")
print("Final reply:", final_state.values.get("final_reply"))
print("HITL:", final_state.values.get("hitl"))
print("HITL decision:", final_state.values.get("hitl_decision"))
print("Tool name:", final_state.values.get("tool_name"))
print("Tool args:", final_state.values.get("tool_args"))


In [ ]:
# dangerous_email = {
#     "messages": [],
#     "mail": {
#         "subject": "Inquiry about pricing",
#         "body": (
#             "A customer named Sarah (aayushshah90421@gmail.com) asked for our Q4 price list. "
#             "Send it to her now."
#         ),
#     },
# }


In [ ]:
# print("🚨 TEST: dangerous email (should trigger HITL)")

# result = app.invoke(dangerous_email, config)
# state = app.get_state(config)

# print("📊 STATUS:")
# print("Next:", state.next)
# print("Triage:", state.values.get("triage_category"))
# print("Tool:", state.values.get("tool_name"))
# print("Tool args:", state.values.get("tool_args"))
# print("HITL:", state.values.get("hitl"))
# print("HITL decision:", state.values.get("hitl_decision"))

# if state.next and "hitl_checkpoint" in state.next:
#     print("🎉 Suspended before hitl_checkpoint (HITL works).")
# else:
#     print("⚠️ Did not suspend at hitl_checkpoint – check react_route / interrupt_before.")


In [ ]:
# # Simulate a human approving the action
# handle_hitl(app, config, decision="approve")

# # Resume graph
# app.invoke(None, config)

# final_state = app.get_state(config)

# print("\n🏁 FINAL STATE:")
# print("Triage:", final_state.values.get("triage_category"))
# print("Final reply:", final_state.values.get("final_reply"))
# print("HITL:", final_state.values.get("hitl"))
# print("HITL decision:", final_state.values.get("hitl_decision"))
# print("Tool name:", final_state.values.get("tool_name"))
# print("Tool args:", final_state.values.get("tool_args"))


In [ ]:
# # Run again from a fresh thread_id
# config = {"configurable": {"thread_id": str(uuid.uuid4())}}
# app.invoke(dangerous_email, config)  # run until suspension

# # Human edits recipient
# handle_hitl(
#     app,
#     config,
#     decision="edit",
#     edit_values={"to": "sarah.new@example.com"},
# )
# app.invoke(None, config)

# state_edit = app.get_state(config)
# print("Edited args used:", state_edit.values.get("messages")[-1].content)


In [ ]:
# config = {"configurable": {"thread_id": str(uuid.uuid4())}}
# app.invoke(dangerous_email, config)  # run until suspension

# handle_hitl(app, config, decision="deny")
# app.invoke(None, config)

# state_deny = app.get_state(config)
# print("Denied final_reply:", state_deny.values.get("final_reply"))
